# Predicting Valorant Esports Map Outcomes
### Model 2: K-Nearest Neighbors (KNN)
**DS 4400** | Rohil Singh, Sean Tian | TA: Matthew Laws

## Step 1 - Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')

print('All imports loaded.')

## Step 2 - Load Data
Upload `VLRDataDummies.csv` and `VLRDataDropped.csv` to Colab.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df_dummies = pd.read_csv('VLRDataDummies.csv').drop(columns=['Unnamed: 0'])
df_dropped = pd.read_csv('VLRDataDropped.csv').drop(columns=['Unnamed: 0'])

print(f'Dummies: {df_dummies.shape[0]:,} records, {df_dummies.shape[1]} columns')
print(f'Dropped: {df_dropped.shape[0]:,} records, {df_dropped.shape[1]} columns')
df_dummies.head()

## Step 3 - Train/Test Split and Scaling
Stratified 80/20 split. Scaling is especially important for KNN since it uses distance between points. Without scaling, features with larger ranges would dominate the distance calculation.

In [ ]:
# Dummies dataset
X_dum = df_dummies.drop(columns=['Team1 Win'])
y_dum = df_dummies['Team1 Win']

X_train_dum, X_test_dum, y_train_dum, y_test_dum = train_test_split(
    X_dum, y_dum, test_size=0.2, random_state=42, stratify=y_dum
)

scaler_dum = StandardScaler()
X_train_dum_s = scaler_dum.fit_transform(X_train_dum)
X_test_dum_s = scaler_dum.transform(X_test_dum)

# Dropped dataset
X_drop = df_dropped.drop(columns=['Team1 Win'])
y_drop = df_dropped['Team1 Win']

X_train_drop, X_test_drop, y_train_drop, y_test_drop = train_test_split(
    X_drop, y_drop, test_size=0.2, random_state=42, stratify=y_drop
)

scaler_drop = StandardScaler()
X_train_drop_s = scaler_drop.fit_transform(X_train_drop)
X_test_drop_s = scaler_drop.transform(X_test_drop)

print(f'Dummies - Train: {X_train_dum.shape[0]:,}, Test: {X_test_dum.shape[0]:,}')
print(f'Dropped - Train: {X_train_drop.shape[0]:,}, Test: {X_test_drop.shape[0]:,}')
print(f'Features: {X_train_dum.shape[1]}')

---
# K-Nearest Neighbors (KNN)
KNN classifies a new data point by finding the k most similar training examples (nearest neighbors) and predicting the majority class among them. The key hyperparameter is k: how many neighbors to check.

We need to find the best k by testing a range of values and picking the one with the highest cross-validation accuracy.

### Step 4 - Finding the Best k
We test odd values of k from 1 to 25 using 5-fold cross-validation on the training set.

**Note:** This cell may take a few minutes to run because KNN computes distances to all training points for every prediction.

In [ ]:
def find_best_k(X_train_s, y_train, dataset_name, k_values=range(1, 26, 2)):
    """Test a range of k values and return results."""
    k_results = []
    for k in k_values:
        knn = KNeighborsClassifier(n_neighbors=k)
        cv_scores = cross_val_score(knn, X_train_s, y_train, cv=5, scoring='accuracy')
        k_results.append((k, cv_scores.mean(), cv_scores.std()))
        print(f'  k={k:2d}: CV Accuracy = {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

    best_k, best_acc, best_std = max(k_results, key=lambda x: x[1])
    print(f'\n  Best k = {best_k} (CV Accuracy = {best_acc:.4f})')
    return k_results, best_k

print('Helper function defined.')

In [ ]:
print('=== Dummies Dataset ===')
k_results_dum, best_k_dum = find_best_k(X_train_dum_s, y_train_dum, 'Dummies')

print('\n=== Dropped Dataset ===')
k_results_drop, best_k_drop = find_best_k(X_train_drop_s, y_train_drop, 'Dropped')

### k Selection Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (k_results, best_k, name) in zip(axes,
    [(k_results_dum, best_k_dum, 'Dummies'), (k_results_drop, best_k_drop, 'Dropped')]):
    ks = [r[0] for r in k_results]
    accs = [r[1] for r in k_results]
    stds = [r[2] for r in k_results]

    ax.errorbar(ks, accs, yerr=stds, fmt='o-', color='#5B8FB9', linewidth=2,
                capsize=4, capthick=1.5, markersize=6)
    ax.axvline(x=best_k, color='red', linestyle='--', linewidth=1.5,
               label=f'Best k = {best_k}')
    ax.set_xlabel('Number of Neighbors (k)')
    ax.set_ylabel('5-Fold CV Accuracy')
    ax.set_title(f'KNN: Accuracy vs k ({name})')
    ax.legend()
    ax.set_xticks(ks)

plt.tight_layout()
plt.show()

## Step 5 - Train and Evaluate with Best k

In [ ]:
def evaluate_knn(X_train_s, X_test_s, y_train, y_test, best_k, dataset_name):
    """Train KNN with best k and evaluate."""
    model = KNeighborsClassifier(n_neighbors=best_k)
    model.fit(X_train_s, y_train)

    pred = model.predict(X_test_s)
    proba = model.predict_proba(X_test_s)[:, 1]

    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred)
    rec = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    auc = roc_auc_score(y_test, proba)

    cv_scores = cross_val_score(model, X_train_s, y_train, cv=5, scoring='accuracy')

    print(f'=== KNN Results (k={best_k}, {dataset_name}) ===')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1-Score:  {f1:.4f}')
    print(f'ROC-AUC:   {auc:.4f}')
    print(f'\n5-Fold CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
    print(f'\nClassification Report:')
    print(classification_report(y_test, pred, target_names=['Loss', 'Win']))

    return model, pred, proba, {'Accuracy': acc, 'Precision': prec, 'Recall': rec,
                                 'F1': f1, 'ROC-AUC': auc, 'CV Mean': cv_scores.mean()}

print('Helper function defined.')

### Dummies Dataset

In [ ]:
knn_dum, pred_dum, proba_dum, results_dum = evaluate_knn(
    X_train_dum_s, X_test_dum_s, y_train_dum, y_test_dum, best_k_dum, 'Dummies'
)

### Dropped Dataset

In [ ]:
knn_drop, pred_drop, proba_drop, results_drop = evaluate_knn(
    X_train_drop_s, X_test_drop_s, y_train_drop, y_test_drop, best_k_drop, 'Dropped'
)

### Side-by-Side Comparison

In [ ]:
comparison = pd.DataFrame({'Dummies': results_dum, 'Dropped': results_drop})
print(comparison.to_string())

## Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (pred, y_test, best_k, name) in zip(axes,
    [(pred_dum, y_test_dum, best_k_dum, 'Dummies'),
     (pred_drop, y_test_drop, best_k_drop, 'Dropped')]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Loss', 'Win'], yticklabels=['Loss', 'Win'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'KNN (k={best_k}) Confusion Matrix ({name})')

plt.tight_layout()
plt.show()

## ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for proba, y_test, best_k, name, color in [
    (proba_dum, y_test_dum, best_k_dum, 'Dummies', '#5B8FB9'),
    (proba_drop, y_test_drop, best_k_drop, 'Dropped', '#E07A5F')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{name} k={best_k} (AUC = {auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Baseline (AUC = 0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve - KNN')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

---
## Summary
KNN performs well but slightly below Logistic Regression on this dataset. This makes sense because KNN is a non-parametric model that relies on local neighborhoods, while Logistic Regression captures global linear relationships across all features. The k-selection plot shows how accuracy changes with different neighbor counts, and the close match between test and CV accuracy confirms the model is not overfitting.